In [1]:
%pip uninstall torch torchvision torchaudio -y

%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Found existing installation: torch 2.7.1+cu118
Uninstalling torch-2.7.1+cu118:
  Successfully uninstalled torch-2.7.1+cu118
Note: you may need to restart the kernel to use updated packages.


Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (27 kB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl (2817.2 MB)
Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp312-cp312-win_amd64.whl (5.5 MB)
Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl (4.1 MB)

   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score

from torchvision import transforms, models

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4

META_TRAIN = "../Data/processed_meld_emotieff/v1_enet_b2_7_gtprob_gap0p3s/metadata_train.csv"
META_DEV = "../Data/processed_meld_emotieff/dev/metadata_dev.csv"

EMOTIONS = sorted(["Anger","Disgust","Fear","Happiness","Neutral","Sadness","Surprise"])
label2id = {e:i for i,e in enumerate(EMOTIONS)}

cuda


In [4]:
class FaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
        # Preload all images
        print("Loading images...")
        self.images = []
        for path in tqdm(self.df["image_path"]):
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            self.images.append(img)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = self.images[idx]  # Already loaded
        if self.transform:
            img = self.transform(img)
        label = label2id[self.df.iloc[idx]["emotion_norm"]]
        return img, label

In [5]:
train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [6]:
train_df = pd.read_csv(META_TRAIN)
dev_df = pd.read_csv(META_DEV)

# IMPORTANT — încearcă și fără filtrare pentru baseline
train_df = train_df[train_df["status"] == "ok"]
dev_df = dev_df[dev_df["status"] == "ok"]

# opțional (pentru variantă curată)
# train_df = train_df[(train_df["selection_rank"] == 0) & (train_df["pred_prob"] >= 0.5)]

train_ds = FaceDataset(train_df, train_tf)
dev_ds = FaceDataset(dev_df, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=DEVICE=="cuda")
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=DEVICE=="cuda")

Loading images...


100%|██████████| 48980/48980 [03:56<00:00, 207.48it/s]


Loading images...


100%|██████████| 5413/5413 [00:25<00:00, 208.89it/s]


In [7]:
def get_light_model():
    model = models.mobilenet_v2(pretrained=True)
    
    # înlocuiești classifier
    model.classifier[1] = nn.Linear(model.last_channel, len(EMOTIONS))
    
    return model.to(DEVICE)

In [8]:
def get_heavy_model():
    model = models.resnet50(pretrained=True)
    
    model.fc = nn.Linear(model.fc.in_features, len(EMOTIONS))
    
    return model.to(DEVICE)

In [9]:
def train_model(model, name):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_acc = 0.0

    for epoch in range(EPOCHS):
        # ---- TRAIN ----
        model.train()
        total_loss = 0

        for imgs, labels in tqdm(train_loader):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(imgs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # ---- EVAL ----
        model.eval()
        correct = 0
        total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for imgs, labels in dev_loader:
                imgs = imgs.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(imgs)
                preds = outputs.argmax(dim=1)

                correct += (preds == labels).sum().item()
                total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_acc = correct / total
        val_f1 = f1_score(all_labels, all_preds, average='weighted')

        print(f"[{name}] Epoch {epoch+1}: loss={total_loss:.4f}, val_acc={val_acc:.4f}, val_f1={val_f1:.4f}")

        # 🔥 SAVE BEST
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_acc": best_acc,
                "epoch": epoch,
            }, f"{name}_best.pth")
            print(f"✅ Saved new best model ({best_acc:.4f})")

In [10]:
print(DEVICE)

cuda


In [11]:
light_model = get_light_model()
train_model(light_model, "MobileNetV2 (Light)")

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 1531/1531 [02:56<00:00,  8.66it/s]


[MobileNetV2 (Light)] Epoch 1: loss=2203.9097, val_acc=0.4081, val_f1=0.3250
✅ Saved new best model (0.4081)


100%|██████████| 1531/1531 [02:56<00:00,  8.69it/s]


[MobileNetV2 (Light)] Epoch 2: loss=2010.9946, val_acc=0.4077, val_f1=0.3396


100%|██████████| 1531/1531 [02:56<00:00,  8.70it/s]


...

[MobileNetV2 (Light)] Epoch 8: loss=843.4153, val_acc=0.3514, val_f1=0.3270


100%|██████████| 1531/1531 [02:52<00:00,  8.89it/s]


[MobileNetV2 (Light)] Epoch 9: loss=713.4210, val_acc=0.3288, val_f1=0.3122


100%|██████████| 1531/1531 [02:50<00:00,  8.95it/s]


[MobileNetV2 (Light)] Epoch 10: loss=628.5081, val_acc=0.3222, val_f1=0.3125


In [12]:
heavy_model = get_heavy_model()
train_model(heavy_model, "ResNet50 (Heavy)")

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 1531/1531 [05:28<00:00,  4.65it/s]


[ResNet50 (Heavy)] Epoch 1: loss=2184.4054, val_acc=0.3891, val_f1=0.3355
✅ Saved new best model (0.3891)


100%|██████████| 1531/1531 [05:28<00:00,  4.66it/s]


[ResNet50 (Heavy)] Epoch 2: loss=1994.7443, val_acc=0.3754, val_f1=0.3345


100%|██████████| 1531/1531 [05:40<00:00,  4.49it/s]


...

[ResNet50 (Heavy)] Epoch 8: loss=563.2326, val_acc=0.3503, val_f1=0.3219


100%|██████████| 1531/1531 [05:35<00:00,  4.56it/s]


[ResNet50 (Heavy)] Epoch 9: loss=470.1532, val_acc=0.3122, val_f1=0.3064


100%|██████████| 1531/1531 [05:44<00:00,  4.45it/s]


[ResNet50 (Heavy)] Epoch 10: loss=402.4081, val_acc=0.3488, val_f1=0.3267


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_model(model_path, model_name):
    """Evaluate model on dev set and return comprehensive metrics"""
    
    # Load model
    checkpoint = torch.load(model_path, map_location=DEVICE)
    
    if "MobileNet" in model_name:
        model = get_light_model()
    else:
        model = get_heavy_model()
    
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in dev_loader:
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    
    return {
        "model_name": model_name,
        "accuracy": accuracy,
        "f1_weighted": f1_weighted,
        "f1_macro": f1_macro,
        "precision": precision,
        "recall": recall,
        "preds": all_preds,
        "labels": all_labels,
        "confusion_matrix": confusion_matrix(all_labels, all_preds)
    }

# Evaluate both models
light_results = evaluate_model("MobileNetV2 (Light)_best.pth", "MobileNetV2 (Light)")
heavy_results = evaluate_model("ResNet50 (Heavy)_best.pth", "ResNet50 (Heavy)")

# Display results
print("="*70)
print("FINAL MODEL PERFORMANCE COMPARISON")
print("="*70)

for results in [light_results, heavy_results]:
    print(f"\n📊 {results['model_name']}")
    print("-"*70)
    print(f"  Accuracy:        {results['accuracy']:.4f}")
    print(f"  F1 Score (weighted): {results['f1_weighted']:.4f}")
    print(f"  F1 Score (macro):    {results['f1_macro']:.4f}")
    print(f"  Precision:       {results['precision']:.4f}")
    print(f"  Recall:          {results['recall']:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(results['labels'], results['preds'], 
                              target_names=EMOTIONS))

# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, results in enumerate([light_results, heavy_results]):
    sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', 
                xticklabels=EMOTIONS, yticklabels=EMOTIONS,
                ax=axes[idx], cmap='Blues')
    axes[idx].set_title(f"{results['model_name']} - Confusion Matrix")
    axes[idx].set_ylabel("True Label")
    axes[idx].set_xlabel("Predicted Label")

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print(f"Better Model: {light_results['model_name'] if light_results['accuracy'] > heavy_results['accuracy'] else heavy_results['model_name']}")
print(f"  by {abs(light_results['accuracy'] - heavy_results['accuracy']):.4f} accuracy difference")
print("="*70)